# Ejercicio 1: Ingesta del Sistema Exportador

In [1]:
import pandas as pd
import numpy as np

np.random.seed(26)
cantidad_pedidos = 500
fechas_despacho = pd.to_datetime('2025-08-01') + pd.to_timedelta(np.random.randint(0, 60, cantidad_pedidos), unit='D')
fechas_llegada = fechas_despacho + pd.to_timedelta(np.random.randint(2, 25, cantidad_pedidos), unit='D')

datos_bodega = {
    'ID_Remito': np.arange(8000, 8000 + cantidad_pedidos),
    'Varietal': np.random.choice(['Malbec', 'Cabernet Franc', 'Chardonnay', 'Syrah'], cantidad_pedidos),
    'Cajas_Enviadas': np.random.randint(10, 200, cantidad_pedidos),
    'Precio_Por_Caja': np.random.randint(15000, 45000, cantidad_pedidos),
    'Fecha_Despacho': fechas_despacho.astype(str),
    'Fecha_Llegada': fechas_llegada.astype(str)
}

df_bodega = pd.DataFrame(datos_bodega)
df_bodega.to_csv('logistica_bodega.csv', index=False)
print('Archivo logistica_bodega.csv generado exitosamente.')

Archivo logistica_bodega.csv generado exitosamente.


---

# Ejercicio 2: Columnas Calculadas (Matemática Vectorizada)

In [2]:
df = pd.read_csv('logistica_bodega.csv')
print('Archivo logistica_bodega.csv leído correctamente.')

Archivo logistica_bodega.csv leído correctamente.


In [3]:
df['Valor_Total_Envio'] = df['Cajas_Enviadas'] * df['Precio_Por_Caja']
print('Columna Valor_Total_Envio creada.')

Columna Valor_Total_Envio creada.


In [4]:
print('Primeras 3 filas:')
df.head(3)

Primeras 3 filas:


,ID_Remito,Varietal,Cajas_Enviadas,Precio_Por_Caja,Fecha_Despacho,Fecha_Llegada,Valor_Total_Envio
0,8000,Malbec,133,33311,2025-09-23,2025-10-06,4430363
1,8001,Cabernet Franc,199,23451,2025-08-07,2025-08-09,4666749
2,8002,Chardonnay,162,23261,2025-09-18,2025-10-12,3768282


---

# Ejercicio 3: Viajando en el Tiempo (datetime)

In [5]:
df['Fecha_Despacho'] = pd.to_datetime(df['Fecha_Despacho'])
df['Fecha_Llegada'] = pd.to_datetime(df['Fecha_Llegada'])
print('Fechas convertidas a formato datetime.')

Fechas convertidas a formato datetime.


In [6]:
df['Dias_En_Transito'] = (df['Fecha_Llegada'] - df['Fecha_Despacho']).dt.days
print('Columna Dias_En_Transito creada.')

Columna Dias_En_Transito creada.


---

# Ejercicio 4: Reglas de Negocio (apply)

In [7]:
def evaluar_calidad_logistica(dias):
    if dias <= 7:
        return 'Óptimo'
    elif dias <= 15:
        return 'Aceptable'
    else:
        return 'Riesgo de Temperatura'

In [8]:
df['Estado_Calidad'] = df['Dias_En_Transito'].apply(evaluar_calidad_logistica)
print('Columna Estado_Calidad creada con apply().')

Columna Estado_Calidad creada con apply().


In [9]:
print('Conteo de envíos por estado de calidad:')
df['Estado_Calidad'].value_counts()

Conteo de envíos por estado de calidad:


Estado_Calidad
Riesgo de Temperatura    194
Aceptable                180
Óptimo                   126
Name: count, dtype: int64

---

# Ejercicio 5: El Reporte Crítico

In [10]:
mascara = (df['Varietal'] == 'Malbec') & (df['Estado_Calidad'] == 'Riesgo de Temperatura')
print('Máscara booleana múltiple aplicada.')

Máscara booleana múltiple aplicada.


In [11]:
df_malbec_riesgo = df[mascara]
print(f'Envíos de Malbec en riesgo: {len(df_malbec_riesgo)}')

Envíos de Malbec en riesgo: 39


In [12]:
df_malbec_riesgo

,ID_Remito,Varietal,Cajas_Enviadas,Precio_Por_Caja,Fecha_Despacho,Fecha_Llegada,Valor_Total_Envio,Dias_En_Transito,Estado_Calidad
37,8037,Malbec,175,34534,2025-08-13,2025-09-04,6043450,22,Riesgo de Temperatura
42,8042,Malbec,103,25468,2025-08-30,2025-09-16,2623204,17,Riesgo de Temperatura
82,8082,Malbec,182,32280,2025-09-17,2025-10-03,5874960,16,Riesgo de Temperatura
94,8094,Malbec,12,27999,2025-08-24,2025-09-09,335988,16,Riesgo de Temperatura
98,8098,Malbec,123,27678,2025-09-08,2025-09-29,3404394,21,Riesgo de Temperatura
107,8107,Malbec,162,29084,2025-09-27,2025-10-20,4711608,23,Riesgo de Temperatura
115,8115,Malbec,31,35025,2025-09-02,2025-09-20,1085775,18,Riesgo de Temperatura
117,8117,Malbec,89,21811,2025-08-05,2025-08-24,1941179,19,Riesgo de Temperatura
119,8119,Malbec,178,41855,2025-08-04,2025-08-26,7450190,22,Riesgo de Temperatura
120,8120,Malbec,59,20958,2025-09-08,2025-09-25,1236522,17,Riesgo de Temperatura


In [13]:
total_riesgo = df_malbec_riesgo['Valor_Total_Envio'].sum()
print(f'💰 Pesos en Malbec en riesgo por demoras: ${total_riesgo:,.2f}')

💰 Pesos en Malbec en riesgo por demoras: $107,635,705.00
